# Proyecto #2 — Pipeline de Machine Learning
## Notebook 03: creación del pipeline de ingeniería de características

**Punto 2 del enunciado** · Responsable: Alfredo Catú (alfredo.mux@galileo.edu) · Rama `proyecto_2_pipeline_predict`

**Objetivo:** convertir la **receta** del notebook `02_feature_exploration.ipynb` (sección 1.c) en un **pipeline de sklearn**
que aplique automáticamente, y siempre igual, todas las transformaciones, y guardarlo en un archivo `.pkl`.

| Inciso | Qué se hace aquí |
|---|---|
| **2.a** | Proceso de elaboración: qué se prepara antes del pipeline, qué hace cada paso y en qué orden |
| **2.b** | Construcción del pipeline con **sklearn**, incluido un operador **Mapper** propio, ajuste y verificación |
| **2.c** | Archivo **`models/pipeline_preprocesamiento.pkl`**: se guarda, se vuelve a cargar y se comprueba |

**Regla contra la fuga de información:** el pipeline se **ajusta (`fit`) solo con los meses de entrenamiento 0-26**; después
solo **aplica (`transform`)** lo aprendido a validación (27-33) y test (34).

## 0. Configuración

In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from predict_future_sales.config import MODELS_DIR
from predict_future_sales.dataset import cargar_datos, cargar_ventas_limpias
from predict_future_sales.features import (
    LIMITES_OBJETIVO, MES_FIN_ENTRENAMIENTO, MES_TEST, construir_tabla_modelo, crear_mapeos,
)
from predict_future_sales.transformers import (
    CodificadorFrecuencia, ImputadorMedianaPorGrupo, IndicadorNulo, Mapper, TopeCuantil,
)

pd.set_option("display.max_columns", 40)
set_config(display="diagram")  # muestra el pipeline como diagrama
SEMILLA = 42
RUTA_PIPELINE = MODELS_DIR / "pipeline_preprocesamiento.pkl"

2026-11-27 21:31:30.018 | INFO     | predict_future_sales.config:<module>:11 - PROJ_ROOT path is: /Users/alfred/Desktop/predict-future-sales


## 2.a Proceso de elaboración del pipeline de ingeniería de características

### 2.a.1 Qué va antes del pipeline y qué va dentro

Un pipeline de sklearn trabaja **fila por fila**: recibe una tabla y le aplica pasos. Por eso la receta del notebook 02 se
divide en dos partes:

| Parte | Pasos | Por qué ahí |
|---|---|---|
| **Antes del pipeline** (`construir_tabla_modelo()` en `features.py`) | Cuadrícula mensual con ceros, objetivo `y` recortado a [0, 20], los 9 lags, `precio_previo` y `meses_desde_primera_venta` | Necesitan la **historia mes a mes** de cada par (mirar meses anteriores), algo que no se puede calcular mirando una sola fila. Es una función reutilizable: el notebook 05 la usará igual para el mes 34 |
| **Dentro del pipeline** (este notebook) | Mapeos, bandera, imputaciones, tope de outliers, `log1p`, codificación y escalado | Son operaciones por fila, y varias **aprenden valores** de los datos (medianas, topes, frecuencias, medias): deben aprenderse **una sola vez** con los meses 0-26 y guardarse en el `.pkl` |

### 2.a.2 Tabla de entrada

In [2]:
datos_crudos = cargar_datos()
ventas = cargar_ventas_limpias()
tabla = construir_tabla_modelo(ventas, datos_crudos["items"], datos_crudos["test"])

entrenamiento = tabla["date_block_num"] <= MES_FIN_ENTRENAMIENTO
validacion = tabla["date_block_num"].between(MES_FIN_ENTRENAMIENTO + 1, MES_TEST - 1)
prueba = tabla["date_block_num"] == MES_TEST
print(f"Tabla de entrada: {len(tabla):,} filas × {tabla.shape[1]} columnas · {tabla.memory_usage().sum() / 1e6:,.0f} MB")
print(f"Entrenamiento (0-26): {entrenamiento.sum():,} · validación (27-33): {validacion.sum():,} · test (34): {prueba.sum():,}")
display(tabla[tabla["date_block_num"] == 33].head())
print("% de nulos por columna (los rellena el pipeline):")
display((tabla.isna().mean() * 100).round(2).to_frame("pct_nulos").T)

Tabla de entrada: 11,128,050 filas × 17 columnas · 634 MB
Entrenamiento (0-26): 9,299,486 · validación (27-33): 1,614,364 · test (34): 214,200


,date_block_num,shop_id,item_id,item_cnt_month,item_category_id,y,par_lag_1,par_lag_2,par_lag_3,par_lag_12,item_lag_1,tienda_lag_1,tienda_lag_12,categoria_lag_1,categoria_lag_12,precio_previo,meses_desde_primera_venta
10675678,33,45,13315,1.0,47,1.0,0.0,0.0,NaN,NaN,0.023256,0.12881,0.134986,0.282835,NaN,649.000000,2.0
10675679,33,45,13880,1.0,55,1.0,0.0,2.0,4.0,0.0,1.325581,0.12881,0.134986,0.177560,0.212814,225.037033,12.0
10675680,33,45,13881,2.0,55,2.0,1.0,1.0,1.0,2.0,0.790698,0.12881,0.134986,0.177560,0.212814,647.991211,33.0
10675681,33,45,13923,1.0,40,1.0,0.0,0.0,0.0,1.0,0.348837,0.12881,0.134986,0.200956,0.229770,158.952667,33.0
10675682,33,45,14227,1.0,30,1.0,0.0,0.0,0.0,1.0,2.069767,0.12881,0.134986,0.717078,1.207211,95.542320,23.0


% de nulos por columna (los rellena el pipeline):


,date_block_num,shop_id,item_id,item_cnt_month,item_category_id,y,par_lag_1,par_lag_2,par_lag_3,par_lag_12,item_lag_1,tienda_lag_1,tienda_lag_12,categoria_lag_1,categoria_lag_12,precio_previo,meses_desde_primera_venta
pct_nulos,0.0,0.0,0.0,1.92,0.0,1.92,22.46,28.21,33.69,71.79,21.63,4.31,45.03,3.46,41.93,9.15,9.15


**Interpretación:**
- La tabla de entrada tiene **11,128,050 filas y 17 columnas** (~630 MB): 9.3 millones de entrenamiento, 1.6 millones de
  validación y los 214,200 pares de test.
- Los **nulos se dejan a propósito**: los lags (22-72 %), `precio_previo` y `meses_desde_primera_venta` (productos sin
  ventas previas) los rellena el pipeline con los valores aprendidos. `y` es nulo solo en el mes 34 (1.92 % de las filas),
  que es lo que se va a predecir.

### 2.a.3 Pasos del pipeline y su orden

| # | Paso del pipeline | Columnas | Proceso del notebook 02 | ¿Aprende de los datos? |
|---|---|---|---|---|
| 1 | **`Mapper`**: crea `ciudad`, `tipo_tienda` y `grupo_categoria` con diccionarios | `shop_id`, `item_category_id` | Mapeos (1.c) | No (diccionarios fijos) |
| 2 | **`IndicadorNulo`**: bandera `item_sin_historia` | `meses_desde_primera_venta` | Bandera de 1.b.i | No |
| 3 | `SimpleImputer` (constante 0) + `StandardScaler` | 9 lags | i. Imputación numérica · vi. Escalado | Sí: medias y desviaciones |
| 4 | **`ImputadorMedianaPorGrupo`** + **`TopeCuantil`** (p99.9) + `log1p` + `StandardScaler` | `precio_previo` (con `item_category_id`) | i · iv. Outliers · v. Transformación · vi | Sí: medianas por categoría, tope, media y desviación |
| 5 | `SimpleImputer` (constante 0) + `StandardScaler` | `meses_desde_primera_venta` | i · vi | Sí: media y desviación |
| 6 | `SimpleImputer` ("Desconocido") + **`CodificadorFrecuencia`** + `StandardScaler` | `ciudad` | ii. Imputación categórica · iii. Codificación · vi | Sí: frecuencias, media y desviación |
| 7 | `SimpleImputer` ("Desconocido") + `OneHotEncoder` | `grupo_categoria`, `tipo_tienda` | ii · iii | Sí: lista de categorías |
| 8 | Sin cambio (*passthrough*) | `item_sin_historia`, `shop_id`, `item_category_id` | Identificadores exógenos (1.c) | No |

Las columnas que no se nombran (`item_id`, `date_block_num`, `item_cnt_month`, `y`) **se descartan** dentro del pipeline:
`item_id` no entra al modelo (comprobación B del notebook 02) y las otras son llaves o el objetivo.

## 2.b Construcción del pipeline con sklearn

### 2.b.1 Transformadores propios (incluido el `Mapper`)

sklearn no trae todas las operaciones de la receta, así que se programaron cinco **transformadores propios** en
`predict_future_sales/transformers.py`. Siguen la interfaz de sklearn (`fit` aprende, `transform` aplica), por lo que se
combinan con las piezas estándar:

| Transformador | Qué hace | Por qué no se usa una pieza estándar |
|---|---|---|
| `Mapper` | Traduce valores con un diccionario (p. ej. tienda 12 → "En línea") y crea columnas nuevas | Es el operador **Mapper** que menciona el enunciado; sklearn no tiene uno equivalente |
| `IndicadorNulo` | Bandera 0/1 si una columna está vacía | `SimpleImputer(add_indicator=True)` escalaría la bandera junto con la variable |
| `ImputadorMedianaPorGrupo` | Rellena el precio con la mediana **de su categoría** | `SimpleImputer` solo usa una mediana global |
| `TopeCuantil` | Limita valores por encima del percentil 99.9 aprendido | sklearn no trae un recorte de *outliers* aprendido |
| `CodificadorFrecuencia` | Reemplaza la ciudad por su frecuencia en el entrenamiento | sklearn no trae codificación por frecuencia |

**Por qué en un archivo `.py` y no en el notebook:** el `.pkl` guarda *dónde* está cada clase, no su código. Si las clases
se definieran en este notebook, los notebooks 04 y 05 no podrían abrir el `.pkl`.

Ejemplo del `Mapper` con los diccionarios del proyecto (`crear_mapeos()` en `features.py`):

In [3]:
mapeos = crear_mapeos(datos_crudos["shops"], datos_crudos["item_categories"])
for nueva, (origen, diccionario) in mapeos.items():
    print(f"{origen} -> {nueva}: {len(diccionario)} valores, p. ej. {dict(list(diccionario.items())[9:13])}")

ejemplo = pd.DataFrame({"shop_id": [9, 12, 31, 55], "item_category_id": [40, 30, 20, 75]})
display(Mapper(mapeos).fit_transform(ejemplo))

shop_id -> ciudad: 60 valores, p. ej. {9: 'Itinerante', 10: 'Zhukovski', 11: 'Zhukovski', 12: 'En línea'}
shop_id -> tipo_tienda: 60 valores, p. ej. {9: 'evento', 10: 'física', 11: 'física', 12: 'en línea'}
item_category_id -> grupo_categoria: 84 valores, p. ej. {9: 'Servicios y otros', 10: 'Consolas', 11: 'Consolas', 12: 'Consolas'}


,shop_id,item_category_id,ciudad,tipo_tienda,grupo_categoria
0,9,40,Itinerante,evento,Cine
1,12,30,En línea,en línea,Juegos PC
2,31,20,Moscú,física,Juegos de consola
3,55,75,En línea,en línea,Programas


### 2.b.2 Ensamblado del pipeline

Se usan `Pipeline` (pasos en secuencia) y `ColumnTransformer` (aplica un sub-pipeline distinto a cada grupo de columnas).
`set_output(transform="pandas")` hace que la salida conserve los nombres de las columnas.

In [4]:
columnas_lag = [c for c in tabla.columns if "_lag_" in c]

preprocesamiento = ColumnTransformer(
    [
        ("lags", make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), columnas_lag),
        (
            "precio",
            make_pipeline(
                ImputadorMedianaPorGrupo(),
                TopeCuantil(cuantil=0.999),
                FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                StandardScaler(),
            ),
            ["precio_previo", "item_category_id"],
        ),
        ("antiguedad", make_pipeline(SimpleImputer(strategy="constant", fill_value=0), StandardScaler()), ["meses_desde_primera_venta"]),
        (
            "ciudad",
            make_pipeline(SimpleImputer(strategy="constant", fill_value="Desconocido"), CodificadorFrecuencia(), StandardScaler()),
            ["ciudad"],
        ),
        (
            "one_hot",
            make_pipeline(
                SimpleImputer(strategy="constant", fill_value="Desconocido"),
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
            ["grupo_categoria", "tipo_tienda"],
        ),
        ("sin_cambio", "passthrough", ["item_sin_historia", "shop_id", "item_category_id"]),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

pipeline_preprocesamiento = Pipeline(
    [
        ("mapeos", Mapper(mapeos)),
        ("bandera_sin_historia", IndicadorNulo(columna="meses_desde_primera_venta", nombre="item_sin_historia")),
        ("preprocesamiento", preprocesamiento),
    ]
).set_output(transform="pandas")
pipeline_preprocesamiento

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('mapeos', ...), ('bandera_sin_historia', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,mapeos,"{'ciudad': ('shop_id', ...), 'grupo_categoria': ('item_category_id', ...), 'tipo_tienda': ('shop_id', ...)}"
,columna,'meses_desde_primera_venta'
,nombre,'item_sin_historia'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('lags', ...), ('precio', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string wi

### 2.b.3 Ajuste con los meses 0-26 y transformación

`fit` se llama **solo con los meses de entrenamiento**; luego `transform` se aplica a toda la tabla (entrenamiento,
validación y test) con los valores aprendidos.

In [5]:
pipeline_preprocesamiento.fit(tabla[entrenamiento])
X = pipeline_preprocesamiento.transform(tabla)

print(f"Salida: {X.shape[0]:,} filas × {X.shape[1]} columnas · nulos: {int(X.isna().sum().sum())}")
print("Columnas:", list(X.columns))

pasos = pipeline_preprocesamiento.named_steps["preprocesamiento"].named_transformers_
precio = pasos["precio"].named_steps
aprendido = {
    "Mediana global de precio (respaldo)": precio["imputadormedianaporgrupo"].mediana_global_,
    "Categorías con mediana de precio": len(precio["imputadormedianaporgrupo"].medianas_),
    "Tope de precio (p99.9)": precio["topecuantil"].topes_["precio_previo"],
    "Frecuencia de Moscú": pasos["ciudad"].named_steps["codificadorfrecuencia"].frecuencias_["ciudad"]["Moscú"],
    "Categorías one-hot": sum(len(c) for c in pasos["one_hot"].named_steps["onehotencoder"].categories_),
}
display(pd.Series(aprendido, name="valor aprendido (meses 0-26)").to_frame())

Salida: 11,128,050 filas × 32 columnas · nulos: 0
Columnas: ['par_lag_1', 'par_lag_2', 'par_lag_3', 'par_lag_12', 'item_lag_1', 'tienda_lag_1', 'tienda_lag_12', 'categoria_lag_1', 'categoria_lag_12', 'precio_previo', 'meses_desde_primera_venta', 'ciudad_frecuencia', 'grupo_categoria_Accesorios', 'grupo_categoria_Baterías', 'grupo_categoria_Cine', 'grupo_categoria_Consolas', 'grupo_categoria_Juegos PC', 'grupo_categoria_Juegos de consola', 'grupo_categoria_Juegos digitales móviles/MAC', 'grupo_categoria_Libros', 'grupo_categoria_Medios vírgenes', 'grupo_categoria_Música', 'grupo_categoria_Programas', 'grupo_categoria_Regalos', 'grupo_categoria_Servicios y otros', 'grupo_categoria_Tarjetas de pago', 'tipo_tienda_en línea', 'tipo_tienda_evento', 'tipo_tienda_física', 'item_sin_historia', 'shop_id', 'item_category_id']


,valor aprendido (meses 0-26)
Mediana global de precio (respaldo),299.000000
Categorías con mediana de precio,72.000000
Tope de precio (p99.9),14208.242188
Frecuencia de Moscú,0.221627
Categorías one-hot,17.000000


**Interpretación:**
- La salida tiene exactamente las **32 variables** definidas en el notebook 02 (9 lags, 3 numéricas escaladas, 17 one-hot,
  la bandera y 2 identificadores) y **ningún nulo**.
- Valores aprendidos con los meses 0-26: mediana de precio para **72 categorías** (las otras 12 no tienen productos con
  precio previo en entrenamiento y usan la **mediana global de 299**), **tope de precio de 14,208**, frecuencia de Moscú 0.222
  y 17 categorías para el one-hot.
- **Diferencia con el notebook 02:** allí el tope (22,990) y la mediana global (399) se calcularon sobre las ventas originales;
  el pipeline los aprende de **su propia tabla de entrada** (los precios previos de la cuadrícula de entrenamiento). Es el
  comportamiento estándar de sklearn y hace que el `.pkl` sea autosuficiente: en los notebooks 04 y 05 no hace falta pasarle
  las ventas originales. El efecto en el resultado se mide en 2.b.4.

### 2.b.4 Verificación

Se comprueba que el pipeline reproduce la receta del notebook 02:
1. **Sin fuga:** las variables escaladas tienen media 0 y desviación 1 **en el entrenamiento** (lo aprendido viene de ahí).
2. **Mismo resultado:** se entrena la misma regresión lineal y el mismo modelo de árboles del notebook 02 (meses 0-26 →
   validación 27-33) y se compara el RMSE.

In [6]:
escaladas = [c for c in X.columns if c in columnas_lag or c in ("precio_previo", "meses_desde_primera_venta", "ciudad_frecuencia")]
display(
    pd.DataFrame(
        {
            "media_entrenamiento": X.loc[entrenamiento, escaladas].mean(),
            "desv_entrenamiento": X.loc[entrenamiento, escaladas].std(),
            "media_validacion": X.loc[validacion, escaladas].mean(),
        }
    ).round(3)
)

y = tabla["y"].to_numpy()
filas_entrena = np.flatnonzero(entrenamiento.to_numpy())
filas_valida = validacion.to_numpy()


def rmse(modelo):
    prediccion = modelo.predict(X[filas_valida]).clip(*LIMITES_OBJETIVO)
    return float(np.sqrt(np.mean((prediccion - y[filas_valida]) ** 2)))


lineal = LinearRegression().fit(X.iloc[filas_entrena], y[filas_entrena])
arboles = HistGradientBoostingRegressor(max_iter=200, random_state=SEMILLA).fit(X.iloc[filas_entrena], y[filas_entrena])
display(
    pd.DataFrame(
        {"Notebook 02 (receta)": [0.8730, 0.8210], "Notebook 03 (pipeline)": [rmse(lineal), rmse(arboles)]},
        index=["Regresión lineal", "Árboles (HistGradientBoosting)"],
    ).round(4)
)
del lineal, arboles

,media_entrenamiento,desv_entrenamiento,media_validacion
par_lag_1,0.0,1.0,-0.023
par_lag_2,0.0,1.0,-0.010
par_lag_3,0.0,1.0,0.010
par_lag_12,-0.0,1.0,0.124
item_lag_1,-0.0,1.0,-0.034
tienda_lag_1,0.0,1.0,-0.121
tienda_lag_12,-0.0,1.0,0.546
categoria_lag_1,0.0,1.0,-0.070
categoria_lag_12,-0.0,1.0,0.289
precio_previo,0.0,1.0,0.249


,Notebook 02 (receta),Notebook 03 (pipeline)
Regresión lineal,0.873,0.8730
Árboles (HistGradientBoosting),0.821,0.8237


**Interpretación:**
- **Sin fuga:** las 12 variables escaladas tienen media 0 y desviación 1 en el entrenamiento, porque todo se aprendió de los
  meses 0-26. En validación las medias se desplazan igual que en el notebook 02 (antigüedad 1.12, `tienda_lag_12` 0.55): es
  el cambio en el tiempo ya analizado, que se maneja reentrenando con los meses 0-33 en el notebook 04.
- **Mismo resultado que la receta:** la regresión lineal da **exactamente el mismo RMSE (0.8730)** y el modelo de árboles
  **0.8237 frente a 0.8210** (+0.3 %), por el tope de precio aprendido de la tabla (14,208 en vez de 22,990). La diferencia es
  mínima y el pipeline queda autosuficiente, por lo que se mantiene.

### 2.b.5 Robustez: datos con vacíos y valores desconocidos

En el entrenamiento las columnas categóricas no tienen vacíos (en el notebook 02 se forzaron para aplicar la imputación).
Aquí se comprueba que el pipeline **ya ajustado** responde bien si en el futuro llegan datos incompletos: se toman 5 filas
del mes 34 y se les ponen **una tienda que no existe (99)**, una **categoría que no existe (999)** y precios y lags vacíos.

In [7]:
muestra = tabla[prueba].head(5).astype({"shop_id": "int16", "item_category_id": "int16"})
muestra.loc[muestra.index[0], "shop_id"] = 99
muestra.loc[muestra.index[1], "item_category_id"] = 999
muestra.loc[muestra.index[2], ["precio_previo", "par_lag_1"]] = np.nan

resultado = pipeline_preprocesamiento.transform(muestra)
columnas_ver = ["ciudad_frecuencia", "precio_previo", "par_lag_1"] + [c for c in resultado.columns if c.startswith("tipo_tienda_")]
display(Mapper(mapeos).fit_transform(muestra)[["shop_id", "item_category_id", "ciudad", "tipo_tienda", "grupo_categoria"]])
display(resultado[columnas_ver].round(3))
print("Nulos en la salida:", int(resultado.isna().sum().sum()))

,shop_id,item_category_id,ciudad,tipo_tienda,grupo_categoria
10913850,99,19,NaN,NaN,Juegos de consola
10913851,5,999,Vólogda,física,NaN
10913852,5,19,Vólogda,física,Juegos de consola
10913853,5,23,Vólogda,física,Juegos de consola
10913854,5,20,Vólogda,física,Juegos de consola


,ciudad_frecuencia,precio_previo,par_lag_1,tipo_tienda_en línea,tipo_tienda_evento,tipo_tienda_física
10913850,-0.940,1.490,-0.235,0.0,0.0,0.0
10913851,-0.684,-0.168,-0.235,0.0,0.0,1.0
10913852,-0.684,1.252,-0.235,0.0,0.0,1.0
10913853,-0.684,1.253,-0.235,0.0,0.0,1.0
10913854,-0.684,2.118,-0.235,0.0,0.0,1.0


Nulos en la salida: 0


**Interpretación:**
- **Tienda desconocida (99):** el `Mapper` no la encuentra, la ciudad y el tipo quedan vacíos, el imputador los marca como
  "Desconocido", la frecuencia de ciudad vale 0 (−0.94 al escalar) y las tres columnas `tipo_tienda_*` quedan en 0.
- **Categoría desconocida (999):** el grupo queda "Desconocido" y todas las columnas `grupo_categoria_*` en 0.
- **Precio y lag vacíos:** el precio toma la mediana de su categoría y el lag vale 0.
- **0 nulos en la salida:** el pipeline no falla con datos incompletos o nuevos, lo que demuestra la imputación categórica
  (proceso ii) funcionando dentro del pipeline.

## 2.c Archivo `.pkl` del pipeline de preprocesamiento

Se guarda el pipeline **ajustado** con `joblib` (la forma recomendada por sklearn para guardar sus objetos) en
`models/pipeline_preprocesamiento.pkl`. El archivo contiene los pasos y los valores aprendidos (medianas, topes,
frecuencias, medias), **no los datos**.

In [8]:
RUTA_PIPELINE.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline_preprocesamiento, RUTA_PIPELINE)
print(f"Guardado: {RUTA_PIPELINE.relative_to(MODELS_DIR.parent)} · {os.path.getsize(RUTA_PIPELINE) / 1024:,.1f} KB")

# Comprobación: se vuelve a cargar y debe producir exactamente la misma salida
pipeline_cargado = joblib.load(RUTA_PIPELINE)
comparar = tabla[prueba]
identico = pipeline_cargado.transform(comparar).equals(X[prueba.to_numpy()])
print(f"¿El pipeline cargado produce exactamente la misma salida para los {len(comparar):,} pares de test? {identico}")

Guardado: models/pipeline_preprocesamiento.pkl · 14.6 KB
¿El pipeline cargado produce exactamente la misma salida para los 214,200 pares de test? True


**Interpretación:**
- El archivo `models/pipeline_preprocesamiento.pkl` pesa **~15 KB**: guarda los pasos y los valores aprendidos, no los datos.
- Al volver a cargarlo, produce **exactamente la misma salida** para los 214,200 pares de test, así que los notebooks 04 y 05
  obtendrán las mismas 32 variables que aquí.

### Cómo usar el `.pkl` en los notebooks 04 y 05

```python
import joblib
from predict_future_sales.config import MODELS_DIR
from predict_future_sales.dataset import cargar_datos, cargar_ventas_limpias
from predict_future_sales.features import construir_tabla_modelo

datos = cargar_datos()
tabla = construir_tabla_modelo(cargar_ventas_limpias(), datos["items"], datos["test"])  # pasos previos
preprocesamiento = joblib.load(MODELS_DIR / "pipeline_preprocesamiento.pkl")             # pipeline ajustado (meses 0-26)
X = preprocesamiento.transform(tabla)                                                    # 32 variables
y = tabla["y"]                                                                           # objetivo (no entra al pipeline)
```

- **Notebook 04:** el modelo ganador se agrega como **paso final**, por ejemplo
  `Pipeline([("preprocesamiento", preprocesamiento), ("modelo", modelo)])`. Para reentrenar con los meses 0-33 (recomendación
  del notebook 02) se puede usar `sklearn.base.clone()` sobre el pipeline completo y volver a ajustarlo.
- **Notebook 05:** basta con `construir_tabla_modelo(...)` y el pipeline completo del notebook 04 sobre las filas del mes 34.

## Conclusiones del punto 2

- **2.a:** la receta del notebook 02 se dividió en **pasos previos** (cuadrícula, objetivo, lags, precio previo y antigüedad,
  en `construir_tabla_modelo()`) y un **pipeline de sklearn** con 8 pasos para las operaciones por fila que aprenden de los
  datos.
- **2.b:** el pipeline se construyó con `Pipeline` y `ColumnTransformer` de sklearn y **cinco transformadores propios**
  (`Mapper`, `IndicadorNulo`, `ImputadorMedianaPorGrupo`, `TopeCuantil`, `CodificadorFrecuencia`) en
  `predict_future_sales/transformers.py`. Cubre los seis procesos: imputación numérica y categórica, codificación, outliers,
  transformación y escalado. Se ajustó solo con los meses 0-26 y reproduce los resultados del notebook 02 (RMSE lineal
  0.8730; árboles 0.8237 frente a 0.8210).
- **2.c:** el pipeline ajustado se guardó en **`models/pipeline_preprocesamiento.pkl`** y se comprobó que, al cargarlo,
  produce exactamente la misma salida.
- **Para el notebook 04:** cargar el `.pkl`, agregar el modelo como paso final y, para la predicción final, reentrenar el
  pipeline completo con los meses 0-33.